## **№1 ИМПОРТЫ БИБЛИОТЕК И ЗАГРУЗКА ФАЙЛА**

In [2]:
import pandas as pd 
import sys
import seaborn as sns
import matplotlib.pyplot as plt

# загрузка датасета из каталога
df = pd.read_csv(r'/Users/antonnevolin/Desktop/stata/full_df/full_df.csv')


pd.set_option("display.width", 120)
pd.set_option('display.max_rows', None)


# вывод названия колонок
print(f'Название колонок в датасете: \n{df.columns}')
print('=' * 100)
print(f'Первые пять строк датасета для первичной оценки правильности: \n{df.head(5)}')


Название колонок в датасете: 
Index(['datetime', 'day_part', 'specials', 'type_news', 'client', 'title', 'type_title', 'sub_type_title',
       'trigger_word', 'full_text', 'editor', 'category', 'group', 'impression_total', 'impression_uniq',
       'views_total', 'views_uniq', 'read_total', 'read_uniq', 'share_total', 'share_uniq', 'avg_time', 'med_time',
       'transition_total', 'transition_uniq'],
      dtype='str')
Первые пять строк датасета для первичной оценки правильности: 
           datetime day_part          specials    type_news         client  \
0   2025-12-01 7:30     утро               все     редакция            МИР   
1   2025-12-01 9:00     утро               все     редакция            МИР   
2  2025-12-01 11:08     день        гинекологи  партнерский  Гедеон Рихтер   
3  2025-12-01 12:10     день  гастроэнтерологи        узкие            МИР   
4  2025-12-01 12:10     день       дерматологи        узкие            МИР   

                                           

## **№2 ОЧИСТКА ДАННЫХ (УДАЛЕНИЕ ПУСТЫХ СТРОК, УДАЛЕНИЕ ДУБЛИКАТОВ)**

In [3]:
# поиск пустых значений
print(f'Суммы пропущенных значений по столбцам: \n{df.isna().sum()}')

# функция для очистки данных (убирать # для очистки требуемого столбца)
def clean_df(df):
    df = df.copy() 
    # очистка от пустых значений
    df['type_title'] = df['type_title'].fillna('')
    df['sub_type_title'] = df['sub_type_title'].fillna('')
    df['trigger_word'] = df['trigger_word'].fillna('')
    df['full_text'] = df['full_text'].fillna('')
    df['editor'] = df['editor'].fillna('')
    df['group'] = df['group'].fillna(0)
    df['impression_total'] = df['impression_total'].fillna(0)
    df['impression_uniq']= df['impression_uniq'].fillna(0)
    df['views_total'] = df['views_total'].fillna(0)
    df['read_total'] = df['read_total'].fillna(0)
    df['read_uniq'] = df['read_uniq'].fillna(0)
    df['share_total'] = df['share_total'].fillna(0)
    df['share_uniq'] = df['share_uniq'].fillna(0)
    df['avg_time'] = df['avg_time'].fillna(0)
    df['med_time'] = df['med_time'].fillna(0)
    df['transition_total'] = df['transition_total'].fillna(0)
    df['transition_uniq'] = df['transition_uniq'].fillna(0)

    return df

clear_df = clean_df(df)

print('=' * 100)
print(f'Проверка очистки столбцов: \n{clear_df.isna().sum()}')

print('=' * 100)
# поиск дубликатов
print(f'Кол-во дубликатов: {df.duplicated().sum()}')
# удаление дубликатов
df.drop_duplicates(inplace=True)



Суммы пропущенных значений по столбцам: 
datetime              0
day_part              0
specials              0
type_news             0
client                0
title                 0
type_title          228
sub_type_title      228
trigger_word        228
full_text             4
editor                7
category              0
group               703
impression_total      4
impression_uniq       3
views_total           1
views_uniq            0
read_total            3
read_uniq             2
share_total          92
share_uniq           89
avg_time              6
med_time              6
transition_total    330
transition_uniq     373
dtype: int64
Проверка очистки столбцов: 
datetime            0
day_part            0
specials            0
type_news           0
client              0
title               0
type_title          0
sub_type_title      0
trigger_word        0
full_text           0
editor              0
category            0
group               0
impression_total    0
impression

## **№3 ПРИВЕДЕНИЕ ДАТЫ И ВРЕМЕНИ К ТРЕБУЕМОМУ ФОРМАТУ**

In [4]:
# функция для приведения времени в требуемый формат (неделя)
def week_datetime(df):
    df = df.copy()
    # приведение столбца к требуемому формату
    df['datetime'] = pd.to_datetime(df['datetime'])
    # приведение к форматам через индекс
    df.set_index('datetime', inplace=True)
    # приведение к году
    df['year'] = df.index.year
    # приведение к месяцу
    df['month'] = df.index.month
    # приведение к дню
    df['day'] = df.index.day
    # приведение к дню недели
    df['day_of_week'] = df.index.day_of_week
    return df

date_df = week_datetime(clear_df)

print(f'Обновленные названия столбцов в датасете (посмотрите в конец списка): \n{date_df.columns}')


Обновленные названия столбцов в датасете (посмотрите в конец списка): 
Index(['day_part', 'specials', 'type_news', 'client', 'title', 'type_title', 'sub_type_title', 'trigger_word',
       'full_text', 'editor', 'category', 'group', 'impression_total', 'impression_uniq', 'views_total', 'views_uniq',
       'read_total', 'read_uniq', 'share_total', 'share_uniq', 'avg_time', 'med_time', 'transition_total',
       'transition_uniq', 'year', 'month', 'day', 'day_of_week'],
      dtype='str')


## **№4 КАТЕГОРИИ: КОЛ-ПОСТОВ, МЕДИАНА + ОБЪЕМЫ ПО АВТОРАМ**

In [22]:
# функция дл сбора основных метрик по медиане
def main_metrics(df):
    # словаь русских метрик
    metrics_ru = {
        'impression_total': 'показы',
        'views_total': 'просмотры',
        'read_total': 'прочтения',
        'share_total': 'поделились',
        'transition_total': 'переходы',
        'title': 'сумма постов',
        'category': 'категория постов',
        'count_symb': 'кол-во символов',
        'editor': 'авторы'
    }
    

    # агрегация по категориям постов
    median_count = df.groupby(['category']).agg({
        'title': 'count',
        'impression_total': 'median',
        'views_total': 'median',
        'read_total': 'median',
        'share_total': 'median',
        'transition_total': 'median'
    }).sort_values(by='title', ascending=False)

    # подсчет медиан маркетинговых метрик и их вывод рядом с медианами основных метрик
    median_count['ctr, %'] = median_count['views_total'] / median_count['impression_total'] * 100
    median_count['rr, %'] = median_count['read_total'] / median_count['views_total'] * 100
    median_count['sr, %'] = median_count['share_total'] / median_count['read_total'] * 100
    median_count['cr, %'] = median_count['read_total'] / median_count['impression_total'] * 100


    # добавление перевода метрик на русский
    median_count = median_count.rename(columns=metrics_ru)

    return median_count

main_metr = main_metrics(date_df)

print(f'Агрегация по медиане для исключения влиния объема контента и аномалий + воронка: \n{main_metr.round(2)}')

Агрегация по медиане для исключения влиния объема контента и аномалий + воронка: 
                       сумма постов   показы  просмотры  прочтения  поделились  переходы  ctr, %  rr, %  sr, %  cr, %
category                                                                                                             
клинический случай              170   2861.0      297.5      209.5         2.0       1.0   10.40  70.42   0.95   7.32
исследования                    151   3789.0      353.0      283.0         4.0       1.0    9.32  80.17   1.41   7.47
медицина                        136    831.0      126.5       67.5         4.0       0.0   15.22  53.36   5.93   8.12
эксклюзив                        75   7381.0      275.0      101.0         8.0       1.0    3.73  36.73   7.92   1.37
задача                           44    354.0       95.0       62.0         1.0       0.0   26.84  65.26   1.61  17.51
здравоохранение                  38   4601.5      752.0      590.0         7.0       2.5   1

In [18]:
# функция для выводов объема по авторам + график
def editors(df):
    # создание словаря для перевода на русский язык
    metrics_ru = {
        'count_symb': 'кол-во символов',
        'editor': 'авторы',
        'title': 'кол-во постов'
    }

    # подсчет символов для передачи в таблицу авторов
    df['count_symb'] = df['full_text'].str.len()
    # агрегация сумм символов по авторам
    symb_editors = df.groupby('editor').agg({
        'title': 'count',
        'count_symb': 'sum'
    }).sort_values(by='count_symb', ascending=False)
    
    # переименования на русский
    symb_editors = symb_editors.rename(columns=metrics_ru)

    return symb_editors

edit_sum = editors(date_df)

print(f'Кол-во символов по авторам: \n{edit_sum}')
print('=' * 100)

Кол-во символов по авторам: 
            кол-во постов  кол-во символов
editor                                    
Антон Н               347           719988
Оля С                 135           327998
Арина Т                96           158327
Вика Б                 64           103240
Лена Т                 18            70783
неизвестно             19            56177
Вероника               11            29726
                        7            20215
Денис Л                 2             7446
Виктория                2             3820
Феруза Н                1             2370
Даша Н                  1              818


## **№5 ВЫВОДЫ ПО БЛОКУ**

**основное:**

## **№6 МАРКЕТИНГОВЫЕ МЕТРИКИ: CTR, RR, SR, CR (ВОРОНКА)**

In [19]:
def mark_metrics_by_category(df):
    # группируем по категории и суммируем нужные столбцы
    grouped = df.groupby(['day_of_week', 'category', 'day_part']).agg(
        sum_impr=('impression_total', 'sum'),
        sum_views=('views_total', 'sum'),
        sum_read=('read_total', 'sum'),
        sum_share=('share_total', 'sum'),
        sum_trans=('transition_total', 'sum')
    )
    
    # считаем производные метрики
    grouped['CTR, %'] = grouped['sum_views'] / grouped['sum_impr'] * 100
    grouped['RR, %'] = grouped['sum_read'] / grouped['sum_views'] * 100
    grouped['SR, %'] = grouped['sum_share'] / grouped['sum_read'] * 100
    grouped['CR, %'] = grouped['sum_trans'] / grouped['sum_read'] * 100
    
    # создаем сводки по медиане по дню недели
    week_marc = grouped.groupby('day_of_week')[['CTR, %', 'RR, %', 'SR, %', 'CR, %']].median()
    # создаем сводки по медиане по категориям
    category_marc = grouped.groupby('category')[['CTR, %', 'RR, %', 'SR, %', 'CR, %']].median()
    # создаем сводки по медиане по времени дня
    day_part_marc = grouped.groupby('day_part')[['CTR, %', 'RR, %', 'SR, %', 'CR, %']].median()

    return week_marc, category_marc, day_part_marc

week_tbl, cat_tbl, part_tbl = mark_metrics_by_category(date_df)

print(f'Воронка потребления контента по дням недели (0=пнд, 6=вскр): \n{week_tbl.round(2)}')
print('='  * 100)
print(f'Воронка потребления контента по времени дня: \n{part_tbl.round(2)}')
print('=' * 100)

Воронка потребления контента по дням недели (0=пнд, 6=вскр): 
             CTR, %  RR, %  SR, %  CR, %
day_of_week                             
0              8.77  67.12   2.59   1.66
1             10.64  59.15   2.19   0.50
2             10.07  52.05   3.17   0.55
3             21.67  42.72   3.09   1.77
4              8.81  54.32   2.67   1.49
5              3.82  83.62   3.22   0.00
6              8.68  34.66   3.82   1.02
Воронка потребления контента по времени дня: 
          CTR, %  RR, %  SR, %  CR, %
day_part                             
вечер      10.58  65.27   2.38   0.81
день        9.60  52.36   3.42   2.50
ночь        5.51  41.45   2.33   0.22
утро        8.27  56.31   1.77   0.17


## **№7 ВЫВОДЫ ПО БЛОКУ**

**основное:**

## **№8 ЗАГОЛОВКИ: ТОП/АНТИТОП**

In [57]:
# функция для определения лучших заголовков (медианы и маркетинговые метрики)
def title_metr(df):
    pass